In [1]:
import torch
from torch import nn, device
from torchvision import transforms, datasets, models
import numpy as np
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
from tqdm.notebook import tqdm

#Transforming the data size to the chosen size
data_transforms = transforms.Compose([transforms.Resize((128,128)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225 ])])

#Checking the environment
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

#Preparing the models
def get_vgg_model(num_classes):
    model = models.vgg16(weights=None)#Weights is zero since we are training from scratch
    in_features = model.classifier[6].in_features
    model.classifier[6] = torch.nn.Linear(model.classifier[6].in_features, num_classes)#The layer to change is the 7th one
    return model

def get_mobilenet_model(num_classes):
    model = models.mobilenet_v2(weights=None)#Weights is zero since we are training from scratch
    in_features = model.classifier[1].in_features
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)#The layer to change is the 2nd one
    return model
#Trainning
def training_model(model, train_loader, val_loader, num_epochs = 5, lr = 0.001):
    #Have to initialize here to avoid potential issues later
    all_predictions = []
    all_labels = []

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr = lr)
    model.to(device)

    for epoch in range(num_epochs):
        #Training code
        model.train()
        running_loss = 0.0

        for images, labels in tqdm(train_loader, desc = f"Epoch {epoch + 1} [Training]"):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        #Validating results
        model.eval()
        val_correct = 0
        val_total = 0
        all_predictions = []
        all_labels = []
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                val_correct += (predicted == labels).sum().item()
                val_total += labels.size(0)
                #For report purposes
                all_labels.extend(labels.cpu().numpy())
                all_predictions.extend(predicted.cpu().numpy())

        val_accuracy = 100 * val_correct / val_total
        print(f"Epoch {epoch + 1} | Loss: {running_loss / len(train_loader): .4f} | Accuracy: {val_accuracy: .2f}%\n")

    #For report purposes
    print("\nFinal Classification Report\n")
    print(classification_report(all_labels, all_predictions))

    return model


PyTorch Version: 2.10.0
GPU Available: False
